# Candle Prediction using Market Depth

In [48]:
from pathlib import Path
import pandas as pd
import numpy as np
from strategies_dev.utils import resample_fractional_minute, apply_trailing_logic, generate_signal

In [49]:
# ---- Input ------
date_ = "27APR2026"
file_name = "NIFTY26APR24000CE.xlsx"

file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\extracted_symbols\{file_name}")
df = pd.read_excel(file_path)
if "PE" in file_name or "CE" in file_name:
    col_name = "last_trade_time"

    # Volume computation
    # 1. Calculate the basic difference between rows
    df["volume_at_tick"] = df["volume_traded"].diff()
    df.loc[df["volume_at_tick"] == 0, "volume_at_tick"] = np.nan
    # df["volume_at_tick"] = df["volume_at_tick"].ffill()
    # df["volume_at_tick"] = df["volume_at_tick"].fillna(0)
else:
    col_name = "local_time"

df[col_name] = pd.to_datetime(df[col_name])
target_date = pd.to_datetime(date_).date()
df = df[df[col_name].dt.date == target_date]

In [50]:
file_name

'NIFTY26APR24000CE.xlsx'

In [51]:
df[["last_trade_time", "last_price", "depth", "volume_traded", "volume_at_tick"]].head(2)

,last_trade_time,last_price,depth,volume_traded,volume_at_tick
1,2026-04-27 09:15:00,172.9,"{'buy': [{'quantity': 455, 'price': 170.65, 'o...",23270,23270.0
2,2026-04-27 09:15:00,161.6,"{'buy': [{'quantity': 455, 'price': 170.65, 'o...",23270,NaN


In [52]:
# ---- Input ------
N = 6
window = 2
price_pct_threshold = 0.001
volume_threshold=1000
volume_period = 30

use_price_pct_level=True
use_price_trend=True
use_volume=False

In [53]:
df.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'last_traded_quantity',
       'average_traded_price', 'option_CE_PE', 'option_type', 'strike',
       'volume_traded', 'total_buy_quantity', 'total_sell_quantity', 'ohlc',
       'change', 'oi', 'oi_day_high', 'oi_day_low', 'depth', 'tradable',
       'mode', 'volume_at_tick'],
      dtype='str')

In [54]:
print(type(df.iloc[0]["local_time"]))
print(df.iloc[0]["local_time"])
print(df.iloc[0]["last_trade_time"])

<class 'pandas.Timestamp'>
2026-04-27 09:15:00.530000
2026-04-27 09:15:00


In [55]:
clubbed_df = resample_fractional_minute(df, col_name, N)
clubbed_df["bucket_time_next"] = clubbed_df["bucket_time"].shift(-1)
clubbed_df["price_diff"] = clubbed_df["close"] - clubbed_df["open"]
clubbed_df["price_pct"] = (clubbed_df["close"] - clubbed_df["open"])/clubbed_df["open"]

clubbed_df["volume_diff"] = clubbed_df["volume_high"] - clubbed_df["volume_low"]
clubbed_df["volume_participated"] = clubbed_df["volume_high"] - clubbed_df["volume_low"]
clubbed_df["volume_pct"] = (clubbed_df["volume_close"] - clubbed_df["volume_open"])/clubbed_df["volume_open"]
clubbed_df["volume_avg"] = clubbed_df["volume_close"].rolling(volume_period).median()

In [56]:
clubbed_df[clubbed_df['bucket_time'].dt.minute == 0].head()

,bucket_time,minute,open,high,low,close,volume_open,volume_high,volume_low,volume_close,...,minute_close,depth_list,ltp_list,bucket_time_next,price_diff,price_pct,volume_diff,volume_participated,volume_pct,volume_avg
270,2026-04-27 10:00:00,2026-04-27 10:00:00,181.60,182.40,179.90,180.00,10790.0,44525.0,10790.0,33085.0,...,180.3,"[{'buy': [{'quantity': 260, 'price': 181.45, '...","[181.6, 182.25, 182.4, 182.3, 180.7, 180.25, 1...",2026-04-27 10:00:10,-1.60,-0.008811,33735.0,33735.0,2.066265,19207.5
271,2026-04-27 10:00:10,2026-04-27 10:00:00,180.10,182.85,180.00,182.85,24960.0,31330.0,12220.0,12220.0,...,180.3,"[{'buy': [{'quantity': 260, 'price': 181.45, '...","[181.6, 182.25, 182.4, 182.3, 180.7, 180.25, 1...",2026-04-27 10:00:20,2.75,0.015269,19110.0,19110.0,-0.510417,18135.0
272,2026-04-27 10:00:20,2026-04-27 10:00:00,183.30,183.55,177.95,178.90,24245.0,59605.0,6500.0,59605.0,...,180.3,"[{'buy': [{'quantity': 260, 'price': 181.45, '...","[181.6, 182.25, 182.4, 182.3, 180.7, 180.25, 1...",2026-04-27 10:00:30,-4.40,-0.024004,53105.0,53105.0,1.458445,19207.5
273,2026-04-27 10:00:30,2026-04-27 10:00:00,179.15,181.90,179.15,179.70,25935.0,75205.0,5330.0,18720.0,...,180.3,"[{'buy': [{'quantity': 260, 'price': 181.45, '...","[181.6, 182.25, 182.4, 182.3, 180.7, 180.25, 1...",2026-04-27 10:00:40,0.55,0.003070,69875.0,69875.0,-0.278195,18622.5
274,2026-04-27 10:00:40,2026-04-27 10:00:00,179.55,180.50,178.00,179.95,9620.0,20215.0,9620.0,17550.0,...,180.3,"[{'buy': [{'quantity': 260, 'price': 181.45, '...","[181.6, 182.25, 182.4, 182.3, 180.7, 180.25, 1...",2026-04-27 10:00:50,0.40,0.002228,10595.0,10595.0,0.824324,18622.5


In [57]:
clubbed_df[["volume_open", "volume_high", "volume_low", "volume_close", "volume_participated"]]

,volume_open,volume_high,volume_low,volume_close,volume_participated
0,23270.0,199745.0,23270.0,168350.0,176475.0
1,167895.0,167895.0,85345.0,132340.0,82550.0
2,112710.0,135850.0,76765.0,96200.0,59085.0
3,130585.0,166595.0,65390.0,65390.0,101205.0
4,102895.0,102895.0,46345.0,76635.0,56550.0
...,...,...,...,...,...
2245,11635.0,22035.0,7800.0,7800.0,14235.0
2246,16965.0,16965.0,10725.0,10725.0,6240.0
2247,13455.0,61880.0,11635.0,26975.0,50245.0
2248,10790.0,174395.0,6955.0,21580.0,167440.0


In [58]:
clubbed_df[clubbed_df['bucket_time'].dt.minute == 0].shape

(36, 24)

In [59]:
# generate_signal
clubbed_df_2 = generate_signal(clubbed_df, window, price_pct_threshold = price_pct_threshold, volume_threshold=volume_threshold,
                               use_price_pct_level=use_price_pct_level, use_price_trend=use_price_trend, use_volume=use_volume)

use_price_pct_level :  True
use_price_trend :  True
use_volume :  False


In [60]:
# clubbed_df_2

In [63]:
# clubbed_df_2[8:11].to_excel("clubbed_df_8_11.xlsx")

In [64]:
clubbed_df_2.columns

Index(['bucket_time', 'minute', 'open', 'high', 'low', 'close', 'volume_open',
       'volume_high', 'volume_low', 'volume_close', 'candle_type',
       'minute_open', 'minute_high', 'minute_low', 'minute_close',
       'depth_list', 'ltp_list', 'bucket_time_next', 'price_diff', 'price_pct',
       'volume_diff', 'volume_participated', 'volume_pct', 'volume_avg',
       'predicted'],
      dtype='str')

In [65]:
clubbed_df_2.shape

(2250, 25)

In [66]:
clubbed_df_2 = clubbed_df[:-1]

In [67]:
clubbed_df_2.shape

(2249, 25)

In [68]:
clubbed_df_2.head(3)

,bucket_time,minute,open,high,low,close,volume_open,volume_high,volume_low,volume_close,...,depth_list,ltp_list,bucket_time_next,price_diff,price_pct,volume_diff,volume_participated,volume_pct,volume_avg,predicted
0,2026-04-27 09:15:00,2026-04-27 09:15:00,172.90,175.05,161.60,168.40,23270.0,199745.0,23270.0,168350.0,...,"[{'buy': [{'quantity': 455, 'price': 170.65, '...","[172.9, 161.6, 170.65, 169.2, 171.3, 164.85, 1...",2026-04-27 09:15:10,-4.50,-0.026027,176475.0,176475.0,6.234637,NaN,NaN
1,2026-04-27 09:15:10,2026-04-27 09:15:00,165.95,174.05,154.45,157.35,167895.0,167895.0,85345.0,132340.0,...,"[{'buy': [{'quantity': 455, 'price': 170.65, '...","[172.9, 161.6, 170.65, 169.2, 171.3, 164.85, 1...",2026-04-27 09:15:20,-8.60,-0.051823,82550.0,82550.0,-0.211769,NaN,NaN
2,2026-04-27 09:15:20,2026-04-27 09:15:00,154.40,159.85,151.55,157.95,112710.0,135850.0,76765.0,96200.0,...,"[{'buy': [{'quantity': 455, 'price': 170.65, '...","[172.9, 161.6, 170.65, 169.2, 171.3, 164.85, 1...",2026-04-27 09:15:30,3.55,0.022992,59085.0,59085.0,-0.146482,NaN,NaN


In [69]:
pwd

'D:\\Study\\Programs\\trading\\strategies_dev'

In [78]:
clubbed_df.columns

Index(['bucket_time', 'minute', 'open', 'high', 'low', 'close', 'volume_open',
       'volume_high', 'volume_low', 'volume_close', 'candle_type',
       'minute_open', 'minute_high', 'minute_low', 'minute_close',
       'depth_list', 'ltp_list', 'bucket_time_next', 'price_diff', 'price_pct',
       'volume_diff', 'volume_participated', 'volume_pct', 'volume_avg',
       'predicted'],
      dtype='str')

In [88]:
import pandas_ta as ta
clubbed_df['roc_10'] = ta.roc(clubbed_df['close'], length=10)
clubbed_df['roc_7'] = ta.roc(clubbed_df['close'], length=7)
clubbed_df['roc_5'] = ta.roc(clubbed_df['close'], length=5)
clubbed_df['roc_3'] = ta.roc(clubbed_df['close'], length=3)

clubbed_df['acc_10'] = clubbed_df["roc_10"].diff()
clubbed_df['acc_10'] = clubbed_df['acc_10'] / clubbed_df['acc_10'].rolling(20).std()

clubbed_df['acc_7'] = clubbed_df["roc_7"].diff()
clubbed_df['acc_7'] = clubbed_df['acc_7'] / clubbed_df['acc_7'].rolling(20).std()

clubbed_df['acc_5'] = clubbed_df["roc_5"].diff()
clubbed_df['acc_5'] = clubbed_df['acc_5'] / clubbed_df['acc_5'].rolling(20).std()

clubbed_df['acc_3'] = clubbed_df["roc_3"].diff()
clubbed_df['acc_3'] = clubbed_df['acc_3'] / clubbed_df['acc_3'].rolling(20).std()

In [89]:
clubbed_df[["bucket_time", "minute", "open", "high", "low", "close", "roc_10","roc_7", "roc_5", "roc_3", "acc_10", "acc_7", "acc_5", "acc_3",  "candle_type", "predicted"]].to_excel("clubbed_df.xlsx")

In [71]:
# df_with_signal = df.merge(
#         clubbed_df[["bucket_time", "predicted"]],
#         left_on="last_trade_time",
#         right_on="bucket_time",
#         how="left"
#     )

# import pandas as pd

# 1. Ensure both DataFrames are sorted by the time columns
df = df.sort_values("last_trade_time")
clubbed_df_2 = clubbed_df_2.sort_values("bucket_time")

# 2. Perform the proximity merge
df_with_signal = pd.merge_asof(
    df,
    clubbed_df_2[["bucket_time_next", "predicted"]],
    left_on="last_trade_time",
    right_on="bucket_time_next",
    direction="backward" # Only looks at the past/current, never the future
)

# Find duplicates in bucket_time and set their 'predicted' value to NaN
df_with_signal.loc[df_with_signal.duplicated(subset=['bucket_time_next'], keep='first'), 'predicted'] = np.nan

In [72]:
# df_with_signal.to_excel("df_with_signal.xlsx")

In [73]:
# clubbed_df_2.to_excel("clubbed_df_2.xlsx")

In [74]:
df_with_signal.head()

,instrument_token,symbol,exchange_timestamp,local_time,last_trade_time,last_price,last_traded_quantity,average_traded_price,option_CE_PE,option_type,...,change,oi,oi_day_high,oi_day_low,depth,tradable,mode,volume_at_tick,bucket_time_next,predicted
0,18499330,NIFTY26APR24000CE,NaN,2026-04-27 09:15:00.530,2026-04-27 09:15:00,172.90,65,171.17,CE,atm_plus_1.0,...,27.366483,8560825,8560825,8560825,"{'buy': [{'quantity': 455, 'price': 170.65, 'o...",True,full,23270.0,NaT,NaN
1,18499330,NIFTY26APR24000CE,NaN,2026-04-27 09:15:01.030,2026-04-27 09:15:00,161.60,390,171.17,CE,atm_plus_1.0,...,19.042357,8560825,8560825,8560825,"{'buy': [{'quantity': 455, 'price': 170.65, 'o...",True,full,NaN,NaT,NaN
2,18499330,NIFTY26APR24000CE,NaN,2026-04-27 09:15:02.030,2026-04-27 09:15:01,170.65,65,170.04,CE,atm,...,25.709024,8560825,8560825,8560825,"{'buy': [{'quantity': 260, 'price': 170.75, 'o...",True,full,83395.0,NaT,NaN
3,18499330,NIFTY26APR24000CE,NaN,2026-04-27 09:15:02.281,2026-04-27 09:15:01,169.20,65,170.04,CE,atm,...,24.640884,8560825,8560825,8560825,"{'buy': [{'quantity': 260, 'price': 170.75, 'o...",True,full,NaN,NaT,NaN
4,18499330,NIFTY26APR24000CE,NaN,2026-04-27 09:15:03.280,2026-04-27 09:15:01,171.30,65,170.04,CE,atm,...,26.187845,8560825,8560825,8560825,"{'buy': [{'quantity': 260, 'price': 170.75, 'o...",True,full,NaN,NaT,NaN


In [75]:
df_with_signal.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'last_traded_quantity',
       'average_traded_price', 'option_CE_PE', 'option_type', 'strike',
       'volume_traded', 'total_buy_quantity', 'total_sell_quantity', 'ohlc',
       'change', 'oi', 'oi_day_high', 'oi_day_low', 'depth', 'tradable',
       'mode', 'volume_at_tick', 'bucket_time_next', 'predicted'],
      dtype='str')

In [76]:
# params = {
#     "initial_sl_pct": 0.02,
#     "target_pct": 0.01,
#     "trail_sl_pct": 0.02,
#     "tight_sl_offset": 0.5,
# }

params = {
    "initial_sl_pct": 0.02,
    "target_pct": 0.001,
    "trail_sl_pct": 0.018,
    "tight_sl_offset": 0.5,
}

In [33]:
trades = apply_trailing_logic(df_with_signal, params)

trades = pd.DataFrame(trades)
if len(trades):
    trades["final"] = trades.apply(lambda row: "profit" if row["profit"] > 0 else "loss", axis=1)
else:
    print("trades not generated")

In [34]:
len(trades)

91

In [35]:
# trades

## Profit

In [36]:
trades[trades["final"]=="profit"]["profit"].sum(), trades[trades["final"]=="profit"]["pnl"].sum()

(np.float64(57.549999999999926), np.float64(21384.72272034988))

### Loss

In [37]:
trades[trades["final"]=="loss"]["profit"].sum(), trades[trades["final"]=="loss"]["pnl"].sum()

(np.float64(-30.220700000000136), np.float64(-28961.825330566156))

### Final

In [38]:
trades[trades["final"]=="profit"]["pnl"].sum() + trades[trades["final"]=="loss"]["pnl"].sum()

np.float64(-7577.102610216276)

In [39]:
trades.head(11)

,entry_time,entry_price,exit_time,exit_price,profit,profit_pct,pnl,final
0,2026-04-23 09:15:31,250.45,2026-04-23 09:15:38,245.9419,-4.5081,-0.018000,-3264.448896,loss
1,2026-04-23 09:22:10,218.25,2026-04-23 09:22:11,220.3500,2.1000,0.009622,1063.298683,profit
2,2026-04-23 09:34:00,218.80,2026-04-23 09:34:02,218.9000,0.1000,0.000457,-235.874243,profit
3,2026-04-23 09:36:10,220.35,2026-04-23 09:36:16,221.5000,1.1500,0.005219,444.060333,profit
4,2026-04-23 09:36:30,224.00,2026-04-23 09:36:30,223.8500,-0.1500,-0.000670,-404.218254,loss
5,2026-04-23 09:38:10,220.40,2026-04-23 09:38:11,220.8000,0.4000,0.001815,-42.948412,profit
6,2026-04-23 09:40:31,214.40,2026-04-23 09:40:36,215.5000,1.1000,0.005131,418.493318,profit
7,2026-04-23 09:40:40,216.05,2026-04-23 09:40:40,216.1000,0.0500,0.000231,-265.150231,profit
8,2026-04-23 09:45:11,200.20,2026-04-23 09:45:13,200.1000,-0.1000,-0.000500,-344.169383,loss
9,2026-04-23 09:55:00,179.25,2026-04-23 09:55:03,176.0235,-3.2265,-0.018000,-2349.822738,loss


In [938]:
trades["final"].value_counts()

final
profit    57
loss      34
Name: count, dtype: int64

In [939]:
trades["final"].value_counts(normalize=True) * 100

final
profit    62.637363
loss      37.362637
Name: proportion, dtype: float64

In [940]:
trades[trades["final"]=="loss"].head(12)

,entry_time,entry_price,exit_time,exit_price,profit,profit_pct,pnl,final
0,2026-04-23 09:15:31,250.45,2026-04-23 09:15:38,245.9419,-4.5081,-0.018000,-3264.448896,loss
4,2026-04-23 09:36:30,224.00,2026-04-23 09:36:30,223.8500,-0.1500,-0.000670,-404.218254,loss
8,2026-04-23 09:45:11,200.20,2026-04-23 09:45:13,200.1000,-0.1000,-0.000500,-344.169383,loss
9,2026-04-23 09:55:00,179.25,2026-04-23 09:55:03,176.0235,-3.2265,-0.018000,-2349.822738,loss
10,2026-04-23 09:56:40,177.35,2026-04-23 09:56:41,177.2000,-0.1500,-0.000846,-350.148385,loss
12,2026-04-23 10:00:51,178.60,2026-04-23 10:00:56,178.5500,-0.0500,-0.000280,-286.670431,loss
15,2026-04-23 10:15:41,180.70,2026-04-23 10:15:43,180.5000,-0.2000,-0.001107,-386.494602,loss
16,2026-04-23 10:21:00,173.25,2026-04-23 10:21:00,173.1000,-0.1500,-0.000866,-345.396264,loss
18,2026-04-23 10:25:10,179.00,2026-04-23 10:25:12,178.7500,-0.2500,-0.001397,-416.987597,loss
19,2026-04-23 10:34:10,193.75,2026-04-23 10:34:11,193.5500,-0.2000,-0.001032,-401.620257,loss


In [941]:
trades[trades["final"]=="loss"].columns

Index(['entry_time', 'entry_price', 'exit_time', 'exit_price', 'profit',
       'profit_pct', 'pnl', 'final'],
      dtype='str')

In [942]:

# clubbed_df_2.to_excel(Path(f"assets/logs/{date_}/clubbed_df.xlsx"))

In [943]:
# clubbed_df_2["predicted"].value_counts()